# Error Handling & Logging - try/except, Custom Errors, and the logging Module

In production data pipelines, things go wrong - 

- File not found
- API timeout
- Database connection failure
- Schema mismatch

If you don't handle these gracefully, your pipeline crashes or silently produces bad data.
That's why error handling and logging are the twin pillars of resilient data engineering.

###  1. The try / except Block
Basic structure to handle errors:

In [0]:
try:
    # risky code
    value = int("abc")
except ValueError as e:
    print(" Error occurred:", e)

Multiple Except Blocks

In [0]:
try:
    file = open("data.csv", "r")
    data = file.read()
except FileNotFoundError:
    print(" File not found. Please check the path.")
except PermissionError:
    print(" Permission denied.")
except Exception as e:
    print(" Unexpected error:", e)

Catch specific exceptions first, and then use a generic Exception block for unknown issues.
Helps pinpoint what kind of failure happened.

### 2. The else and finally Clauses

In [0]:
try:
    result = 10 / 2
except ZeroDivisionError:
    print("Cannot divide by zero.")
else:
    print(" Division successful:", result)
finally:
    print(" Cleanup complete.")

### 3. Raising Custom Exceptions
When you want to enforce rules (like schema checks), raise your own error.

In [0]:
def validate_salary(salary):
    if salary < 0:
        raise ValueError("Salary cannot be negative!")
    return salary

try:
    validate_salary(-100)
except ValueError as e:
    print(" Data Validation Error:", e)

### 4 Logging - The Professional Way to Monitor Pipelines
Printing errors is fine for testing.
 But in production, we log everything to a file - with timestamps, severity levels, and details.

In [0]:
import logging

logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Pipeline started")
logging.warning("API took longer than expected")
logging.error("File missing in input folder")

### 5. Combining Error Handling and Logging
Let's combine everything into one practical pipeline example.

In [0]:
import requests, pandas as pd, logging
from datetime import datetime

# Setup logging
logging.basicConfig(
    filename="api_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

def fetch_api_data(url):
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        logging.info(" Successfully fetched API data.")
        return response.json()
    except requests.exceptions.Timeout:
        logging.error(" Timeout while calling API.")
    except requests.exceptions.HTTPError as e:
        logging.error(f" HTTP Error: {e}")
    except Exception as e:
        logging.exception(" Unexpected error during API fetch.")
    return None

def process_data(data):
    try:
        df = pd.DataFrame(data)
        if df.empty:
            raise ValueError("DataFrame is empty after fetch.")
        logging.info(f" Loaded {len(df)} records.")
        return df
    except ValueError as e:
        logging.error(f"Data Validation Error: {e}")
    except Exception as e:
        logging.exception(" Error during data processing.")
    return pd.DataFrame()

def save_data(df):
    try:
        df.to_csv("/Volumes/thedataengineering_00/data/sampledata/output.csv", index=False)
        logging.info(" Data successfully written to output.csv.")
    except Exception as e:
        logging.exception(" Error writing file.")

# Main pipeline flow
if __name__ == "__main__":
    logging.info(" Pipeline started.")
    data = fetch_api_data("https://jsonplaceholder.typicode.com/users")
    if data:
        df = process_data(data)
        if not df.empty:
            save_data(df)
    logging.info(" Pipeline completed successfully.")

###  6. Logging to Console and File Together
You can log both to console and file using StreamHandler.

In [0]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# File handler
fh = logging.FileHandler("pipeline.log")
# Console handler
ch = logging.StreamHandler()

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
ch.setFormatter(formatter)

logger.addHandler(fh)
logger.addHandler(ch)

logger.info("Logging to both console and file!")

![](/Volumes/thedataengineering_00/data/sampledata/sampleimages/python_error_exception.png)

# Summary
With good error handling and logging, your code moves from working to production-ready.

Practice Tasks

- Write a try/except block that handles division by zero.
- Create a custom InvalidSalaryError exception and raise it when salary < 0.
- Log messages at all levels (DEBUG → CRITICAL).
- Build a mini API fetch pipeline with proper logging.
- Add an intentional error and verify that it's logged properly.